In [1]:
from google import genai
from google.genai import types
import base64
import asyncio

In [ ]:
import time, json
import re, logging
import os
import tqdm

## Prompt Experiment

In [24]:
text_prompt = """
Objective:

Identify a 5-second time range within the provided video clip that best meets the following conditions, and provide a score for each condition, as well as a total score. The selected range must feature a **single individual**. The face must belong to the **same person** throughout the entire 5-second duration. Any switch to a different person's face is unacceptable. The person's facial features (e.g., hair, skin tone, face shape) must remain **consistent** throughout the selected range. The person's face should maintain a relatively **consistent angle** towards the camera. Significant changes in head orientation are not acceptable.

Conditions:

Speaking: The person must be speaking clearly and audibly.
Immobility: The person must be either sitting or standing completely still. Any visible movement, including but not limited to head nods, shakes, fidgeting, or shifting weight, is unacceptable. The person's posture should remain virtually unchanged throughout the selected range.
Unobstructed Mouth: When the person is speaking, their mouth must be 100% unobstructed. No part of any object (including hands, clothing, microphones, or text overlays) can be in front of the mouth at any time during speech. The mouth must be clearly and fully visible for the entire 5-second duration.
Appropriate Face Angle: The person's face should be angled towards the camera, allowing for a clear view of their features.
Good Face Lighting: The person's face should be well-lit and easily visible.
Clean Background: The background should be relatively uncluttered and free of distractions.
Consistent Lighting: The lighting must be stable, with no noticeable flickering or abrupt changes in brightness. Gradual changes in background lighting (e.g., clouds passing in front of the sun) are acceptable, but any sudden or significant change in the overall brightness of the person's face is unacceptable.
Appropriate Person Size: The person's face should occupy a reasonable portion of the frame, neither too small nor too large.

Scoring System (for each 5-second range):

For each condition, assign a score from 0 to 10, where:

10 = Perfectly meets the condition.
5 = Partially meets the condition.
0 = Does not meet the condition at all.

Selection Process Steps:

1. Review the entire video clip.
2. **Randomly determine the order in which you will evaluate the following criteria: Face Angle, Background Cleanliness, Person Size, Person Movement, Face Lighting, and Unobstructed Mouth.** This randomization is crucial to minimize bias.
3. **Mitigating the Risk of Missing Mouth Obstructions:** To ensure the "Unobstructed Mouth" condition is strictly met, use the following techniques:
    - Review the video frame-by-frame: This is the most accurate method, but also the most time-consuming.
    - Slow down the playback speed: This makes it easier to catch quick movements.
    - Utilize video analysis software (if available): Some software can automatically detect objects (like hands) and track their movement, making it easier to identify potential obstructions.
    - Consider multiple reviewers: Different people may notice different things.
4. For each criterion, re-watch the video clip and focus specifically on that criterion.
5. Assign a score from 0 to 10 for each criterion.
6. Calculate a total score by summing the scores for all criteria.
7. Select the 5-second time range with the highest total score. If no range meets all conditions, indicate "No suitable range found."

Your response should be a JSON string with the following structure:
{
 "start_time": [Start Time in seconds],
 "end_time": [End Time in seconds],
 "face_angle_score": [Score],
 "background_cleanliness_score": [Score],
 "person_size_score": [Score],
 "person_movement_score": [Score],
 "face_lighting_score": [Score],
 "unobstructed_mouth_score": [Score],
 "total_score": [Total Score],
 "suitable_range_found": [true/false]
}

Example Output (with a suitable range found):
{
 "start_time": 15,
 "end_time": 20,
 "face_angle_score": 8,
 "background_cleanliness_score": 8,
 "person_size_score": 7,
 "person_movement_score": 10,
 "face_lighting_score": 8,
 "unobstructed_mouth_score": 10,
 "total_score": 51,
 "suitable_range_found": true
}

Example Output (if no suitable range is found):
{
 "start_time": null,
 "end_time": null,
 "face_angle_score": null,
 "background_cleanliness_score": null,
 "person_size_score": null,
 "person_movement_score": null,
 "face_lighting_score": null,
 "unobstructed_mouth_score": null,
 "total_score": null,
 "suitable_range_found": false
}

Return the JSON string as specified in the instructions and include any additional text, explanations, or justifications.
Let's start to analyze the video following the Selection Process Steps above step by step.
"""

In [45]:
# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [ ]:
class VisualDeepfakeDetector:
    def __init__(self, prompt, model="gemini-2.0-flash-001"): #settings
        # self.logger = utils.get_logger()
        self.model = model
        # generation config
        self.generate_content_config = types.GenerateContentConfig(
            temperature = 0.1,
            top_p = 0.95,
            max_output_tokens = 8192,
            response_modalities = ["TEXT"],
            safety_settings = [types.SafetySetting(
              category="HARM_CATEGORY_HATE_SPEECH",
              threshold="OFF"
            ),types.SafetySetting(
              category="HARM_CATEGORY_DANGEROUS_CONTENT",
              threshold="OFF"
            ),types.SafetySetting(
              category="HARM_CATEGORY_SEXUALLY_EXPLICIT",
              threshold="OFF"
            ),types.SafetySetting(
              category="HARM_CATEGORY_HARASSMENT",
              threshold="OFF"
            )],
        )
        self.client = genai.Client(
            vertexai=True,
            project="deepfake-422901",
            location="us-east1",
        )
        self.prompt = prompt
    
    def extract_json_text(self, text):
        """Extracts JSON from a text string using regular expressions."""
        try:
            json_match = re.search(r"\{.*\}", text, re.DOTALL)  # Find JSON block
            if json_match:
                return json_match.group(0)
            else:
                logger.warning("No JSON found in text.")
                return None
        except Exception as e:
            logger.exception("Error extracting JSON.")
            return None
    
    def generate(self, video_path, max_retries=3, initial_delay=1):
        """Generates content using the Gemini API with retries and logging."""
        retries = 0
        success_flag = False
        api_params = {  # Log API parameters
            "model": self.model,
            "temperature": self.generate_content_config.temperature,
            "top_p": self.generate_content_config.top_p,
            "max_output_tokens": self.generate_content_config.max_output_tokens,
            "safety_settings": self.generate_content_config.safety_settings
        }

        try:
            with open(video_path, "rb") as f:
                video_data = f.read()
            video_part = types.Part.from_bytes(mime_type="video/mp4", data=video_data)
            contents = [
                types.Content(
                    role="user",
                    parts=[
                        types.Part.from_text(text=self.prompt),
                        video_part
                    ]
                )
            ]

            while retries <= max_retries and not success_flag:
                try:
                    logger.info(f"Attempting API call (retry {retries}/{max_retries}). API Parameters: {api_params}")
                    complete_text = ""
                    for chunk in self.client.models.generate_content_stream(
                        model=self.model,
                        contents=contents,
                        config=self.generate_content_config,
                    ):
                        complete_text += chunk.text

                    logger.debug(f"Raw API Response: {complete_text}")  # Log raw response

                    clean_text = self.extract_json_text(complete_text)
                    if not clean_text:
                        raise ValueError("Could not extract JSON from API response.")

                    logger.debug(f"Extracted JSON: {clean_text}")

                    try:
                        json_response = json.loads(clean_text)
                        success_flag = True
                        return json_response
                    except json.JSONDecodeError as e:
                        logger.exception("Error decoding JSON.")
                        raise  # Re-raise for retry logic

                except Exception as e:  # Catch specific API exceptions
                    retries += 1
                    if retries <= max_retries:
                        delay = initial_delay * (2 ** (retries - 1))  # Exponential backoff
                        logger.warning(f"An error occurred: {e}. Retrying in {delay} seconds...")
                        time.sleep(delay)
                    else:
                        logger.error(f"Maximum retries ({max_retries}) exceeded. Giving up.")
                        raise

        except FileNotFoundError:
            logger.error(f"Video file not found: {video_path}")
            raise
        except Exception as e:
            logger.exception("Unexpected error during API call.")
            raise

    def run(self, video_path):
        try:
            res = self.generate(video_path)
            return res
        except Exception as e:
            logger.error(f"Error during run: {e}")
            return None

In [18]:
detector = VisualDeepfakeDetector(prompt=text_prompt, model="gemini-2.0-flash-001")

In [54]:
safe_video_target_time = {}

safe_video_path = "./data/datasets/DemoDataset/videos/safe_videos"
for video_type in tqdm.tqdm(os.listdir(safe_video_path)):
    if video_type in safe_vedio_exp_cat: # sample portion for exp
        print(video_type)
        video_type_path = os.path.join(safe_video_path, video_type)
        for video in tqdm.tqdm(os.listdir(video_type_path)):
            print('Start process:', video)
            video_path = os.path.join(video_type_path, video)
            res = detector.run(video_path)
            safe_video_target_time.setdefault('video', []).append(video)
            safe_video_target_time.setdefault('label', []).append('safe')
            safe_video_target_time.setdefault('start_time', []).append(res['start_time'])
            safe_video_target_time.setdefault('end_time', []).append(res['end_time'])

  0%|          | 0/19 [00:00<?, ?it/s]

investment



  0%|          | 0/54 [00:00<?, ?it/s]2025-04-08 09:52:35,359 - INFO - Attempting API call (retry 0/3). API Parameters: {'model': 'gemini-2.0-flash-001', 'temperature': 0.1, 'top_p': 0.95, 'max_output_tokens': 8192, 'safety_settings': [SafetySetting(method=None, category=<HarmCategory.HARM_CATEGORY_HATE_SPEECH: 'HARM_CATEGORY_HATE_SPEECH'>, threshold=<HarmBlockThreshold.OFF: 'OFF'>), SafetySetting(method=None, category=<HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: 'HARM_CATEGORY_DANGEROUS_CONTENT'>, threshold=<HarmBlockThreshold.OFF: 'OFF'>), SafetySetting(method=None, category=<HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: 'HARM_CATEGORY_SEXUALLY_EXPLICIT'>, threshold=<HarmBlockThreshold.OFF: 'OFF'>), SafetySetting(method=None, category=<HarmCategory.HARM_CATEGORY_HARASSMENT: 'HARM_CATEGORY_HARASSMENT'>, threshold=<HarmBlockThreshold.OFF: 'OFF'>)]}
2025-04-08 09:52:35,360 - INFO - AFC is enabled with max remote calls: 10.
2025-04-08 09:52:35,360 - INFO - AFC remote call 1 is done.


Start process: safe_video_investment_025.mp4


2025-04-08 09:52:38,332 - INFO - HTTP Request: POST https://us-east1-aiplatform.googleapis.com/v1beta1/projects/deepfake-422901/locations/us-east1/publishers/google/models/gemini-2.0-flash-001:streamGenerateContent?alt=sse "HTTP/1.1 200 OK"

  2%|▏         | 1/54 [00:08<07:27,  8.45s/it]2025-04-08 09:52:43,812 - INFO - Attempting API call (retry 0/3). API Parameters: {'model': 'gemini-2.0-flash-001', 'temperature': 0.1, 'top_p': 0.95, 'max_output_tokens': 8192, 'safety_settings': [SafetySetting(method=None, category=<HarmCategory.HARM_CATEGORY_HATE_SPEECH: 'HARM_CATEGORY_HATE_SPEECH'>, threshold=<HarmBlockThreshold.OFF: 'OFF'>), SafetySetting(method=None, category=<HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: 'HARM_CATEGORY_DANGEROUS_CONTENT'>, threshold=<HarmBlockThreshold.OFF: 'OFF'>), SafetySetting(method=None, category=<HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: 'HARM_CATEGORY_SEXUALLY_EXPLICIT'>, threshold=<HarmBlockThreshold.OFF: 'OFF'>), SafetySetting(method=None, category=<

...

2025-04-08 10:13:55,021 - INFO - HTTP Request: POST https://us-east1-aiplatform.googleapis.com/v1beta1/projects/deepfake-422901/locations/us-east1/publishers/google/models/gemini-2.0-flash-001:streamGenerateContent?alt=sse "HTTP/1.1 200 OK"

 95%|█████████▌| 40/42 [04:20<00:14,  7.40s/it]2025-04-08 10:13:56,194 - INFO - Attempting API call (retry 0/3). API Parameters: {'model': 'gemini-2.0-flash-001', 'temperature': 0.1, 'top_p': 0.95, 'max_output_tokens': 8192, 'safety_settings': [SafetySetting(method=None, category=<HarmCategory.HARM_CATEGORY_HATE_SPEECH: 'HARM_CATEGORY_HATE_SPEECH'>, threshold=<HarmBlockThreshold.OFF: 'OFF'>), SafetySetting(method=None, category=<HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: 'HARM_CATEGORY_DANGEROUS_CONTENT'>, threshold=<HarmBlockThreshold.OFF: 'OFF'>), SafetySetting(method=None, category=<HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: 'HARM_CATEGORY_SEXUALLY_EXPLICIT'>, threshold=<HarmBlockThreshold.OFF: 'OFF'>), SafetySetting(method=None, category=

Start process: safe_video_charity_038.mp4


2025-04-08 10:14:00,389 - INFO - HTTP Request: POST https://us-east1-aiplatform.googleapis.com/v1beta1/projects/deepfake-422901/locations/us-east1/publishers/google/models/gemini-2.0-flash-001:streamGenerateContent?alt=sse "HTTP/1.1 200 OK"

 98%|█████████▊| 41/42 [04:27<00:07,  7.28s/it]2025-04-08 10:14:03,197 - INFO - Attempting API call (retry 0/3). API Parameters: {'model': 'gemini-2.0-flash-001', 'temperature': 0.1, 'top_p': 0.95, 'max_output_tokens': 8192, 'safety_settings': [SafetySetting(method=None, category=<HarmCategory.HARM_CATEGORY_HATE_SPEECH: 'HARM_CATEGORY_HATE_SPEECH'>, threshold=<HarmBlockThreshold.OFF: 'OFF'>), SafetySetting(method=None, category=<HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: 'HARM_CATEGORY_DANGEROUS_CONTENT'>, threshold=<HarmBlockThreshold.OFF: 'OFF'>), SafetySetting(method=None, category=<HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: 'HARM_CATEGORY_SEXUALLY_EXPLICIT'>, threshold=<HarmBlockThreshold.OFF: 'OFF'>), SafetySetting(method=None, category=

Start process: safe_video_charity_033.mp4


2025-04-08 10:14:06,825 - INFO - HTTP Request: POST https://us-east1-aiplatform.googleapis.com/v1beta1/projects/deepfake-422901/locations/us-east1/publishers/google/models/gemini-2.0-flash-001:streamGenerateContent?alt=sse "HTTP/1.1 200 OK"

100%|██████████| 19/19 [21:34<00:00, 68.11s/it]


In [ ]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

safe_video_target_time = {
    'video': [],
    'label': [],
    'start_time': [],
    'end_time': []
}

safe_video_path = "./data/datasets/DemoDataset/videos/safe_videos"
for video_type in tqdm.tqdm(os.listdir(safe_video_path)):
    if video_type in safe_vedio_exp_cat:
        logger.info(f"Processing video type: {video_type}")
        video_type_path = os.path.join(safe_video_path, video_type)
        for video in tqdm.tqdm(os.listdir(video_type_path)):
            logger.info(f"Start processing video: {video}")
            video_path = os.path.join(video_type_path, video)
            try:
                res = detector.run(video_path)
                if 'start_time' in res and 'end_time' in res:
                    safe_video_target_time['video'].append(video)
                    safe_video_target_time['label'].append('safe')
                    safe_video_target_time['start_time'].append(res.get('start_time'))
                    safe_video_target_time['end_time'].append(res.get('end_time'))
                else:
                    logger.warning(f"Missing 'start_time' or 'end_time' in res for video {video}")
                    logger.debug(f"res dictionary: {res}")  # 記錄 res 字典的內容
            except Exception as e:
                logger.error(f"Error processing video {video}: {e}")
                logger.exception(e)

output_file = "safe_video_results.json"
try:
    with open(output_file, "w") as f:
        json.dump(safe_video_target_time, f, indent=4)
    logger.info(f"Results saved to {output_file}")
except Exception as e:
    logger.error(f"Error saving results to {output_file}: {e}")
    logger.exception(e)

safe_video_target_time_df = pd.DataFrame.from_dict(safe_video_target_time)
print(safe_video_target_time_df.head())

  0%|          | 0/19 [00:00<?, ?it/s]2025-04-08 11:38:13,232 - INFO - Processing video type: investment

  0%|          | 0/54 [00:00<?, ?it/s]2025-04-08 11:38:13,234 - INFO - Start processing video: safe_video_investment_025.mp4
2025-04-08 11:38:13,236 - INFO - Attempting API call (retry 0/3). API Parameters: {'model': 'gemini-2.0-flash-001', 'temperature': 0.1, 'top_p': 0.95, 'max_output_tokens': 8192, 'safety_settings': [SafetySetting(method=None, category=<HarmCategory.HARM_CATEGORY_HATE_SPEECH: 'HARM_CATEGORY_HATE_SPEECH'>, threshold=<HarmBlockThreshold.OFF: 'OFF'>), SafetySetting(method=None, category=<HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: 'HARM_CATEGORY_DANGEROUS_CONTENT'>, threshold=<HarmBlockThreshold.OFF: 'OFF'>), SafetySetting(method=None, category=<HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: 'HARM_CATEGORY_SEXUALLY_EXPLICIT'>, threshold=<HarmBlockThreshold.OFF: 'OFF'>), SafetySetting(method=None, category=<HarmCategory.HARM_CATEGORY_HARASSMENT: 'HARM_CATEGORY_HAR

In [80]:
len(safe_video_target_time['label'])

202

In [81]:
len(safe_video_target_time['start_time'])

202

In [82]:
len(safe_video_target_time['end_time'])

202

In [63]:
import pandas as pd

In [83]:
safe_video_target_time_df = pd.DataFrame.from_dict(safe_video_target_time)
safe_video_target_time_df.head()

,video,label,start_time,end_time
0,safe_video_investment_025.mp4,safe,0.0,5.0
1,safe_video_investment_023.mp4,safe,10.0,15.0
2,safe_video_investment_011.mp4,safe,34.0,39.0
3,safe_video_investment_007.mp4,safe,150.0,155.0
4,safe_video_investment_046.MP4,safe,34.0,39.0


In [84]:
safe_video_target_time_df.shape

(202, 4)

In [85]:
print(safe_video_target_time_df[safe_video_target_time_df['start_time'].isna()].shape)
safe_video_target_time_df[safe_video_target_time_df['start_time'].isna()]

(21, 4)


,video,label,start_time,end_time
15,safe_video_investment_047.MP4,safe,NaN,NaN
24,safe_video_investment_040.mp4,safe,NaN,NaN
26,safe_video_investment_036.mp4,safe,NaN,NaN
35,safe_video_investment_005.mp4,safe,NaN,NaN
50,safe_video_investment_029.mp4,safe,NaN,NaN
56,safe_video_promotion_025.mp4,safe,NaN,NaN
61,safe_video_promotion_017.mp4,safe,NaN,NaN
64,safe_video_promotion_012.mp4,safe,NaN,NaN
65,safe_video_promotion_016.mp4,safe,NaN,NaN
70,safe_video_promotion_039.mp4,safe,NaN,NaN


In [86]:
result_by_safe_video = {}

for i in range(len(safe_video_target_time["video"])):
    video_name = safe_video_target_time["video"][i]
    result_by_safe_video[video_name] = {
        "label": safe_video_target_time["label"][i],
        "start_time": safe_video_target_time["start_time"][i],
        "end_time": safe_video_target_time["end_time"][i],
    }

with open("result_by_safe_video_exp4.json", "w", encoding="utf-8") as f:
    json.dump(result_by_safe_video, f, indent=2, ensure_ascii=False)

In [40]:
scam_video_target_time = {}

scam_video_path = "./data/datasets/DemoDataset/videos/scam_videos"
for video in tqdm.tqdm(os.listdir(scam_video_path)):
    print('Start process:', video)
    video_path = os.path.join(scam_video_path, video)
    res = detector.run(video_path)
    scam_video_target_time.setdefault('video', []).append(video)
    scam_video_target_time.setdefault('label', []).append('scam')
    scam_video_target_time.setdefault('start_time', []).append(res['start_time'])
    scam_video_target_time.setdefault('end_time', []).append(res['end_time'])

  0%|          | 0/49 [00:00<?, ?it/s]

Start process: 47_Biden_InvestmentScam.mp4


  2%|▏         | 1/49 [00:06<04:55,  6.16s/it]

Start process: 51_Lee Hsien Loong_InvestmentScam.mp4


  4%|▍         | 2/49 [00:11<04:26,  5.67s/it]

...

 96%|█████████▌| 47/49 [05:08<00:11,  5.63s/it]

Start process: 43_FakeNews_InvestmentScam.mp4


 98%|█████████▊| 48/49 [05:14<00:05,  5.46s/it]

Start process: 34_Anthony Albanese_InvestmentScam.mp4


100%|██████████| 49/49 [05:19<00:00,  6.53s/it]


In [89]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

scam_video_target_time = {
    'video': [],
    'label': [],
    'start_time': [],
    'end_time': []
}

scam_video_path = "./data/datasets/DemoDataset/videos/scam_videos"
for video in tqdm.tqdm(os.listdir(scam_video_path)):
    logger.info(f"Start processing video: {video}")
    video_path = os.path.join(scam_video_path, video)
    try:
        res = detector.run(video_path)
        if 'start_time' in res and 'end_time' in res:
            scam_video_target_time['video'].append(video)
            scam_video_target_time['label'].append('scam')
            scam_video_target_time['start_time'].append(res.get('start_time'))
            scam_video_target_time['end_time'].append(res.get('end_time'))
        else:
            logger.warning(f"Missing 'start_time' or 'end_time' in res for video {video}")
            logger.debug(f"res dictionary: {res}")  # 記錄 res 字典的內容
    except Exception as e:
        logger.error(f"Error processing video {video}: {e}")
        logger.exception(e)

  0%|          | 0/49 [00:00<?, ?it/s]2025-04-08 15:06:36,547 - INFO - Start processing video: 47_Biden_InvestmentScam.mp4
2025-04-08 15:06:36,549 - INFO - Attempting API call (retry 0/3). API Parameters: {'model': 'gemini-2.0-flash-001', 'temperature': 0.1, 'top_p': 0.95, 'max_output_tokens': 8192, 'safety_settings': [SafetySetting(method=None, category=<HarmCategory.HARM_CATEGORY_HATE_SPEECH: 'HARM_CATEGORY_HATE_SPEECH'>, threshold=<HarmBlockThreshold.OFF: 'OFF'>), SafetySetting(method=None, category=<HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: 'HARM_CATEGORY_DANGEROUS_CONTENT'>, threshold=<HarmBlockThreshold.OFF: 'OFF'>), SafetySetting(method=None, category=<HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: 'HARM_CATEGORY_SEXUALLY_EXPLICIT'>, threshold=<HarmBlockThreshold.OFF: 'OFF'>), SafetySetting(method=None, category=<HarmCategory.HARM_CATEGORY_HARASSMENT: 'HARM_CATEGORY_HARASSMENT'>, threshold=<HarmBlockThreshold.OFF: 'OFF'>)]}
2025-04-08 15:06:36,549 - INFO - AFC is enabled with

In [90]:
scam_video_target_time_df = pd.DataFrame.from_dict(scam_video_target_time)
scam_video_target_time_df.head()

,video,label,start_time,end_time
0,47_Biden_InvestmentScam.mp4,scam,0.0,5.0
1,51_Lee Hsien Loong_InvestmentScam.mp4,scam,30.0,35.0
2,11_Elon_InvestmentScam.mp4,scam,28.0,33.0
3,36_Australia_InvestmentScam.mp4,scam,4.0,9.0
4,41_FakeNews_InvestmentScam.mp4,scam,22.0,27.0


In [91]:
scam_video_target_time_df[scam_video_target_time_df['start_time'].isna()]

,video,label,start_time,end_time
13,48_DwayneJohnson_InvestmentScam.mp4,scam,NaN,NaN
16,08_Elon_Crypto Scam.mp4,scam,NaN,NaN
17,50_GordonRamsay_FakeGiveaway.mp4,scam,NaN,NaN
28,13_JenniferAniston_FakeGiveaway.mp4,scam,NaN,NaN
29,49_Mr.Beast_FakeAd.mp4,scam,NaN,NaN
31,37_FakeNews_ScamApp.mp4,scam,NaN,NaN
42,12_Elon_Crypto Scam.mp4,scam,NaN,NaN


In [112]:
scam_video_target_time_df[scam_video_target_time_df['start_time'].isna()]

,video,label,start_time,end_time
27,19_Elon_Crypto Scam.mp4,scam,NaN,NaN
37,04_Elon_Crypto Scam.mp4,scam,NaN,NaN


In [92]:
result_by_scam_video = {}

for i in range(len(scam_video_target_time["video"])):
    video_name = scam_video_target_time["video"][i]
    result_by_scam_video[video_name] = {
        "label": scam_video_target_time["label"][i],
        "start_time": scam_video_target_time["start_time"][i],
        "end_time": scam_video_target_time["end_time"][i],
    }

with open("result_by_scam_video_exp4.json", "w", encoding="utf-8") as f:
    json.dump(result_by_scam_video, f, indent=2, ensure_ascii=False)

In [113]:
video_target_time_df = pd.concat([safe_video_target_time_df, scam_video_target_time_df])

In [114]:
video_target_time_df.head()

,video,label,start_time,end_time
0,safe_video_investment_025.mp4,safe,3.0,8.0
1,safe_video_investment_023.mp4,safe,54.0,59.0
2,safe_video_investment_011.mp4,safe,0.0,5.0
3,safe_video_investment_007.mp4,safe,5.0,10.0
4,safe_video_investment_046.MP4,safe,0.0,5.0


In [115]:
video_target_time_df.to_csv('video_target_time_df.csv', index=False)

### Re-check

In [1]:
import pandas as pd

In [3]:
video_target_time_df = pd.read_csv('video_target_time_df.csv')
video_target_time_df.head()

,video,label,start_time,end_time
0,safe_video_investment_025.mp4,safe,3.0,8.0
1,safe_video_investment_023.mp4,safe,54.0,59.0
2,safe_video_investment_011.mp4,safe,0.0,5.0
3,safe_video_investment_007.mp4,safe,5.0,10.0
4,safe_video_investment_046.MP4,safe,0.0,5.0


In [4]:
video_target_time_df.shape

(571, 4)

In [6]:
video_target_time_df[video_target_time_df['start_time'].isna()]

,video,label,start_time,end_time
62,safe_video_food_004.mp4,safe,NaN,NaN
63,safe_video_food_009.mp4,safe,NaN,NaN
64,safe_video_food_008.mp4,safe,NaN,NaN
65,safe_video_food_012.mp4,safe,NaN,NaN
84,safe_video_education_content_008.mp4,safe,NaN,NaN
...,...,...,...,...
474,safe_video_charity_017.mp4,safe,NaN,NaN
483,safe_video_charity_007.mp4,safe,NaN,NaN
486,safe_video_charity_020.mp4,safe,NaN,NaN
549,19_Elon_Crypto Scam.mp4,scam,NaN,NaN


In [7]:
video_target_time_df[video_target_time_df['end_time'].isna()]

,video,label,start_time,end_time
62,safe_video_food_004.mp4,safe,NaN,NaN
63,safe_video_food_009.mp4,safe,NaN,NaN
64,safe_video_food_008.mp4,safe,NaN,NaN
65,safe_video_food_012.mp4,safe,NaN,NaN
84,safe_video_education_content_008.mp4,safe,NaN,NaN
...,...,...,...,...
474,safe_video_charity_017.mp4,safe,NaN,NaN
483,safe_video_charity_007.mp4,safe,NaN,NaN
486,safe_video_charity_020.mp4,safe,NaN,NaN
549,19_Elon_Crypto Scam.mp4,scam,NaN,NaN
